<a href="https://colab.research.google.com/github/jamexhuang/mkt2026group/blob/master/Final_report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

中文註解：此區塊為分析流程說明與章節標題。

In [ ]:
# 中文註解：讀取 Excel 資料並載入為 DataFrame。

import pandas as pd

data = pd.read_excel('/content/bookingcom.xlsx')

In [ ]:
# 中文註解：執行此儲存格的資料處理與特徵工程步驟。

import re

data["url"] = data["tweet_content"].str.count(r"https://t\.co/\w+")

In [ ]:
# 中文註解：計算每則貼文中的 hashtag 數量。

data["hashtag"] = data["tweet_content"].str.count(r"#\w+")

In [ ]:
# 中文註解：安裝本儲存格所需的套件，並進行後續特徵計算。

!pip install emoji
import emoji

data["emoji"] = data["tweet_content"].apply(lambda x: emoji.emoji_count(str(x)))

In [ ]:
# 中文註解：移除網址後，計算問號數量。

data["question"] = data["tweet_content"].str.replace(r"https?://\S+", "", regex=True).str.count(r"\?")

In [ ]:
# 中文註解：計算每則貼文中的 @ 提及次數。

data["at_mention"] = data["tweet_content"].str.count(r"@\w+")

In [ ]:
# 中文註解：移除網址後，計算驚嘆號數量。

data["exclamation"] = data["tweet_content"].str.replace(r"https?://\S+", "", regex=True).str.count(r"!")

In [ ]:
# 中文註解：進行初步文字清洗（移除網址、標註與特殊符號）。

import re

data["text_clean1"] = data["tweet_content"] \
    .str.replace(r"https?://\S+", "", regex=True) \
    .str.replace(r"@\w+", "", regex=True) \
    .str.replace(r"#\w+", "", regex=True) \
    .str.replace(r"[^\w\s,?.!]", "", regex=True)

In [ ]:
# 中文註解：進行初步文字清洗（移除網址、標註與特殊符號）。

data["text_clean2"] = data["text_clean1"].str.lower()

In [ ]:
# 中文註解：安裝本儲存格所需的套件，並進行後續特徵計算。

!pip install contractions
import contractions

data["text_clean3"] = data["text_clean2"].apply(lambda x: contractions.fix(str(x)))

In [ ]:
# 中文註解：展開英文縮寫，提升文字可讀性。

data["text_clean4"] = data["text_clean3"].str.replace(r"[^a-zA-Z\s]", "", regex=True)

In [ ]:
# 中文註解：移除非英文字母字元，只保留字母與空白。

from nltk.corpus import words
import nltk

nltk.download("words")
english_words = set(words.words())

data["text_clean5"] = data["text_clean4"].apply(lambda x: " ".join([w for w in str(x).split() if w.lower() in english_words]))

[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Package words is already up-to-date!


In [ ]:
# 中文註解：只保留英文字典中的單字，降低雜訊。

data["length"] = data["text_clean5"].str.split().str.len()

In [ ]:
# 中文註解：只保留英文字典中的單字，降低雜訊。

import nltk
from nltk.corpus import stopwords

nltk.download("stopwords")
stop_words = set(stopwords.words("english"))

data["text_clean6"] = data["text_clean5"].apply(lambda x: " ".join([w for w in str(x).split() if w not in stop_words]))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Tokenize and Stem the Words

中文註解：此區塊為分析流程說明與章節標題。

In [ ]:
# 中文註解：移除停用字，保留較具意義的詞彙。

data["tweet_tokenized"] = data["text_clean6"].apply(lambda x: x.split())

In [ ]:
# 中文註解：將文字切詞（tokenize）以利後續分析。

from nltk.stem import PorterStemmer
ps = PorterStemmer()
data["tweet_stemmed"] = data["tweet_tokenized"].apply(lambda tokens:
[ps.stem(t) for t in tokens])

 Count-Based Dictionary Method

中文註解：此區塊為分析流程說明與章節標題。

In [ ]:
# 中文註解：讀取 Excel 資料並載入為 DataFrame。

import pandas as pd
sincerity = pd.read_excel("/content/brand_personality.xlsx",
sheet_name="sincerity")

In [ ]:
# 中文註解：對詞彙做 stemming，合併詞形變化。

word_counts = data["tweet_stemmed"].apply(lambda x: len(x))
unique_word_counts = sorted(word_counts.unique())
unique_word_counts

In [ ]:
# 中文註解：將文字切詞（tokenize）以利後續分析。

data["three_gram"] = data["tweet_tokenized"].apply(lambda tokens:
[tuple(tokens[i:i+3]) for i in range(len(tokens)-2)])
data["two_gram"] = data["tweet_tokenized"].apply(lambda tokens:
[tuple(tokens[i:i+2]) for i in range(len(tokens)-1)])
data["one_gram"] = data["tweet_tokenized"].apply(lambda tokens: tokens)

In [ ]:
# 中文註解：建立 1-gram、2-gram、3-gram 詞組。

sincerity_terms = set(sincerity["term"].astype(str))
def match_terms(row):
    matches = []
    for grams in [row["three_gram"], row["two_gram"], row["one_gram"]]:
        for g in grams:
            term = " ".join(g) if isinstance(g, tuple) else g
            if term in sincerity_terms:
                matches.append(term)
    return matches
data["matched_sincerity_terms"] = data.apply(match_terms, axis=1)
data["matched_sincerity_count"] = data["matched_sincerity_terms"].apply(len)

In [ ]:
# 中文註解：將文字切詞（tokenize）以利後續分析。

data["sincerity"] = data["matched_sincerity_count"] / data["tweet_tokenized"].apply(len)

Score-Based Dictionary

中文註解：此區塊為分析流程說明與章節標題。

In [ ]:
# 中文註解：讀取 Excel 資料並載入為 DataFrame。

import pandas as pd

el = pd.read_excel("/content/evaluative_lexicon.xlsx", sheet_name="Sheet1")

In [ ]:
# 中文註解：執行此儲存格的資料處理與特徵工程步驟。

sorted(el["term"].apply(lambda x: len(str(x).split())).unique())

[np.int64(1), np.int64(2)]

In [ ]:
# 中文註解：建立 1-gram、2-gram、3-gram 詞組。

el_dict = dict(zip(el["term"].astype(str), el["valence"]))
def match_valence(row):
    terms = []
    scores = []
    for grams in [row["two_gram"], row["one_gram"]]:
        for g in grams:
            term = " ".join(g) if isinstance(g, tuple) else g
            if term in el_dict:
                terms.append(term)
                scores.append(el_dict[term])
    return pd.Series([terms, scores])
data[["matched_valence_terms", "matched_valence_scores"]] = data.apply(match_valence, axis=1)

In [ ]:
# 中文註解：計算 valence 指標（以 4.5 為中心標準化）。

data["valence"] = data["matched_valence_scores"].apply(lambda x: sum([s - 4.5
for s in x]) / len(x) if len(x) > 0 else 0)

In [ ]:
# 中文註解：指定 extremity 與 emotionality 欄位。

term_col = el.columns[0]
extremity_col = el.columns[2]
emotionality_col = el.columns[3]

In [ ]:
# 中文註解：將文字切詞（tokenize）以利後續分析。

from nltk.util import ngrams
import pandas as pd

term_col = el.columns[0]
extremity_col = el.columns[2]
emotionality_col = el.columns[3]

extremity_dict = dict(zip(el[term_col].astype(str).str.lower(), el[extremity_col]))
emotionality_dict = dict(zip(el[term_col].astype(str).str.lower(), el[emotionality_col]))

def match_extremity(tokens):
    tokens = [str(t).lower() for t in tokens]
    matched_terms = []
    matched_scores = []

    for n in [3, 2, 1]:
        for gram in ngrams(tokens, n):
            term = " ".join(gram)
            if term in extremity_dict:
                matched_terms.append(term)
                matched_scores.append(extremity_dict[term])

    return pd.Series([matched_terms, matched_scores])

def match_emotionality(tokens):
    tokens = [str(t).lower() for t in tokens]
    matched_terms = []
    matched_scores = []

    for n in [3, 2, 1]:
        for gram in ngrams(tokens, n):
            term = " ".join(gram)
            if term in emotionality_dict:
                matched_terms.append(term)
                matched_scores.append(emotionality_dict[term])

    return pd.Series([matched_terms, matched_scores])

data[["matched_extremity_terms", "matched_extremity_scores"]] = data["tweet_tokenized"].apply(match_extremity)
data[["matched_emotionality_terms", "matched_emotionality_scores"]] = data["tweet_tokenized"].apply(match_emotionality)

In [ ]:
# 中文註解：計算 extremity 與 emotionality 指標。

data["extremity"] = data["matched_extremity_scores"].apply(
    lambda x: sum([score - 4.5 for score in x]) / len(x) if len(x) > 0 else 0
)

data["emotionality"] = data["matched_emotionality_scores"].apply(
    lambda x: sum([score - 4.5 for score in x]) / len(x) if len(x) > 0 else 0
)

In [ ]:
# 中文註解：此儲存格預留給後續分析步驟。


In [ ]:
# 中文註解：建立 CTA 詞表並計算每則貼文 CTA 次數。

import re

cta_words = [
    "check out", "learn more", "book now", "find out", "discover more",
    "explore now", "read more", "click here", "shop now", "sign up",
    "join us", "get started", "see more", "visit now", "plan your trip"
]

pattern = r"\b(?:%s)\b" % "|".join([re.escape(x) for x in cta_words])

data["cta"] = data["tweet_content"].str.lower().str.count(pattern)

In [ ]:
# 中文註解：讀取 Excel 資料並載入為 DataFrame。

import pandas as pd
import re

# Ensure 'data' is loaded, as it was not defined in the previous execution context.
data = pd.read_excel('/content/bookingcom.xlsx')

cta_words = [
    "check out", "learn more", "book now", "find out", "discover more",
    "explore now", "read more", "click here", "shop now", "sign up",
    "join us", "get started", "see more", "visit now", "plan your trip"
]

pattern = r"\b(?:%s)\b" % "|".join([re.escape(x) for x in cta_words])

data["cta"] = data["tweet_content"].str.lower().str.count(pattern)

print("Calculated 'cta' column successfully.")
# You might want to display a few rows to verify
# display(data[['tweet_content', 'cta']].head())

Calculated 'cta' column successfully.


In [ ]:
# 中文註解：此儲存格預留給後續分析步驟。


In [ ]:
# 中文註解：檢視前幾筆資料，確認清洗與特徵欄位。

display(data.head(30))

,id,tt_account,date,tweet_content,like,comment,share,picture,url,hashtag,...,sincerity,matched_valence_terms,matched_valence_scores,valence,matched_extremity_terms,matched_extremity_scores,matched_emotionality_terms,matched_emotionality_scores,extremity,emotionality
0,132870,bookingcom,"21/11/2022, 14:00:08",The Dolomites is said to have the best views i...,16,38,4,2,2,0,...,0.000000,[best],[8.4483],3.948300,[best],[3.9483],[best],[4.0741],-0.551700,-0.42590
1,132871,bookingcom,"21/11/2022, 14:00:05",Due to high altitudes and a whopping 900 snow ...,18,56,5,2,2,0,...,0.000000,"[enjoy, stunning, well]","[8.0667, 7.4643, 7.0]",3.010333,"[enjoy, stunning, well]","[3.5667, 2.9643, 2.5]","[enjoy, stunning, well]","[6.0, 5.7586, 2.5172]",-1.489667,0.25860
2,132872,bookingcom,"21/11/2022, 14:00:02",Planning a ski trip to Italy this season? 🏔️ C...,33,240,13,1,1,0,...,0.000000,[],[],0.000000,[],[],[],[],0.000000,0.00000
3,132873,bookingcom,"20/11/2022, 16:06:24",Skip the park and park it at the hotel. The Wa...,5,20,4,0,1,0,...,0.000000,[tired],[3.0357],-1.464300,[tired],[1.4643],[tired],[3.0],-3.035700,-1.50000
4,132874,bookingcom,"20/11/2022, 16:06:22",The WonderWorks building is upside down. They ...,7,18,3,1,2,0,...,0.000000,[],[],0.000000,[],[],[],[],0.000000,0.00000
5,132875,bookingcom,"20/11/2022, 16:06:13",The best place to take your kids on vacay? Orl...,22,43,6,1,1,0,...,0.000000,[best],[8.4483],3.948300,[best],[3.9483],[best],[4.0741],-0.551700,-0.42590
6,132876,bookingcom,"18/11/2022, 10:51:28","For a change of vibe, stay at The Reverie Saig...",7,17,5,2,2,0,...,0.000000,[feel],[5.2],0.700000,[feel],[0.7],[feel],[6.4194],-3.800000,1.91940
7,132877,bookingcom,"18/11/2022, 10:51:25",Take a trip through the reeds of the Mekong de...,8,16,4,1,2,0,...,0.062500,"[authentic, delicious]","[7.5357, 8.3103]",3.423000,"[authentic, delicious]","[3.0357, 3.8103]","[authentic, delicious]","[3.4483, 4.7037]",-1.077000,-0.42400
8,132878,bookingcom,"18/11/2022, 10:51:22","Ho Chi Minh has its own Notre Dame cathedral, ...",15,22,4,1,1,0,...,0.000000,[],[],0.000000,[],[],[],[],0.000000,0.00000
9,132879,bookingcom,"17/11/2022, 10:00:06",“I will remember this new wireless vacuum for ...,8,22,4,0,2,2,...,0.000000,[],[],0.000000,[],[],[],[],0.000000,0.00000


In [ ]:
# 中文註解：將最終清洗後資料匯出為 Excel。

data.to_excel("Final_Report_data_cleaned.xlsx", index=False)